In [1]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
# data = np.load("radar_features_filtered_manual.npz")
data = np.load("radar_features_filtered.npz")

X = data["X"]
y = data["y"]
u = data["u"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [3]:
# data = np.load("radar_features_filtered_manual.npz")
data = np.load("radar_features_filtered.npz")

X = data["X"]
y = data["y"]
u = data["u"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [5]:
catboost = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    verbose=100
)

catboost.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 1.7597055	test: 1.7717624	best: 1.7717624 (0)	total: 98.9ms	remaining: 1m 38s
100:	learn: 0.5761779	test: 1.0860139	best: 1.0860139 (100)	total: 1.82s	remaining: 16.2s
200:	learn: 0.2646556	test: 0.9110449	best: 0.9109604 (199)	total: 3.22s	remaining: 12.8s
300:	learn: 0.1533258	test: 0.8464127	best: 0.8458330 (299)	total: 4.7s	remaining: 10.9s
400:	learn: 0.1064689	test: 0.8133792	best: 0.8133792 (400)	total: 6.11s	remaining: 9.13s
500:	learn: 0.0789800	test: 0.7945244	best: 0.7945244 (500)	total: 7.52s	remaining: 7.49s
600:	learn: 0.0613362	test: 0.7752339	best: 0.7751520 (579)	total: 8.98s	remaining: 5.96s
700:	learn: 0.0503937	test: 0.7663638	best: 0.7656152 (693)	total: 10.8s	remaining: 4.6s
800:	learn: 0.0422899	test: 0.7615621	best: 0.7604716 (783)	total: 12.4s	remaining: 3.08s
900:	learn: 0.0360989	test: 0.7578706	best: 0.7559243 (877)	total: 13.9s	remaining: 1.52s
999:	learn: 0.0315141	test: 0.7551268	best: 0.7544500 (987)	total: 15.3s	remaining: 0us

bestTest = 0.75

CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.05, loss_function='MultiClass', verbose=100)

In [6]:
val_pred = catboost.predict(X_val)
print("VALIDATION")
print(classification_report(y_val, val_pred))

VALIDATION
              precision    recall  f1-score   support

           0       0.71      0.50      0.59        10
           1       0.86      0.75      0.80         8
           2       0.67      0.95      0.78        21
           3       0.67      0.40      0.50         5
           4       1.00      0.50      0.67         2
           5       0.75      0.50      0.60         6

    accuracy                           0.71        52
   macro avg       0.78      0.60      0.66        52
weighted avg       0.73      0.71      0.70        52



In [7]:
test_pred = catboost.predict(X_test)
print("TEST")
print(classification_report(y_test, test_pred))

TEST
              precision    recall  f1-score   support

           0       0.67      0.60      0.63        10
           1       0.88      0.88      0.88         8
           2       0.75      0.86      0.80        21
           3       0.83      0.83      0.83         6
           4       0.00      0.00      0.00         1
           5       0.67      0.57      0.62         7

    accuracy                           0.75        53
   macro avg       0.63      0.62      0.63        53
weighted avg       0.74      0.75      0.74        53



In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    catboost,
    X,
    y_encoded,
    cv=cv,
    scoring="f1_macro"
)

print("CV scores:", scores)
print("Mean F1:", scores.mean())

0:	learn: 1.7557159	total: 30.4ms	remaining: 30.4s
100:	learn: 0.5903689	total: 2.05s	remaining: 18.3s
200:	learn: 0.2863205	total: 3.65s	remaining: 14.5s
300:	learn: 0.1667378	total: 5.16s	remaining: 12s
400:	learn: 0.1133154	total: 7.02s	remaining: 10.5s
500:	learn: 0.0835181	total: 8.61s	remaining: 8.58s
600:	learn: 0.0652262	total: 10.1s	remaining: 6.72s
700:	learn: 0.0528912	total: 11.8s	remaining: 5.01s
800:	learn: 0.0439644	total: 13.3s	remaining: 3.31s
900:	learn: 0.0375833	total: 14.9s	remaining: 1.63s
999:	learn: 0.0327569	total: 16.5s	remaining: 0us
0:	learn: 1.7579795	total: 40.7ms	remaining: 40.7s
100:	learn: 0.5762555	total: 1.76s	remaining: 15.7s
200:	learn: 0.2695422	total: 3.32s	remaining: 13.2s
300:	learn: 0.1643057	total: 5.04s	remaining: 11.7s
400:	learn: 0.1116040	total: 7.08s	remaining: 10.6s
500:	learn: 0.0823126	total: 8.88s	remaining: 8.84s
600:	learn: 0.0643431	total: 10.6s	remaining: 7.01s
700:	learn: 0.0521464	total: 12.3s	remaining: 5.25s
800:	learn: 0.0433

In [9]:
# Use leave-one-user-out next
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, test_idx = next(gss.split(X, y_encoded, groups=u))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

In [10]:
catboost.fit(X_train, y_train)

0:	learn: 1.7521017	total: 17.2ms	remaining: 17.1s
100:	learn: 0.5507293	total: 1.72s	remaining: 15.3s
200:	learn: 0.2703029	total: 3.35s	remaining: 13.3s
300:	learn: 0.1630409	total: 4.94s	remaining: 11.5s
400:	learn: 0.1119028	total: 6.74s	remaining: 10.1s
500:	learn: 0.0824368	total: 8.67s	remaining: 8.63s
600:	learn: 0.0642745	total: 10.6s	remaining: 7.02s
700:	learn: 0.0520444	total: 12.5s	remaining: 5.32s
800:	learn: 0.0433422	total: 14.1s	remaining: 3.5s
900:	learn: 0.0367878	total: 15.8s	remaining: 1.74s
999:	learn: 0.0320093	total: 17.5s	remaining: 0us


CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.05, loss_function='MultiClass', verbose=100)

In [11]:
test_pred = catboost.predict(X_test)
print("TEST")
print(classification_report(y_test, test_pred))

TEST
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.82      0.93      0.88        15
           2       0.44      1.00      0.61         7
           3       0.75      0.60      0.67         5
           4       0.00      0.00      0.00         5
           5       0.60      0.27      0.38        11

    accuracy                           0.63        43
   macro avg       0.44      0.47      0.42        43
weighted avg       0.60      0.63      0.58        43



In [12]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

logo = LeaveOneGroupOut()

acc_scores = []
f1_scores = []

for train_idx, test_idx in logo.split(X, y_encoded, groups=u):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    catboost.fit(X_train, y_train)

    y_pred = catboost.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    acc_scores.append(acc)
    f1_scores.append(f1)

    print(f"Fold Accuracy: {acc:.4f} | Fold F1: {f1:.4f}")

print("\n========== FINAL ==========")
print(f"Mean Accuracy: {np.mean(acc_scores):.4f}")
print(f"Mean Macro F1: {np.mean(f1_scores):.4f}")
print(f"Std Accuracy: {np.std(acc_scores):.4f}")
print(f"Std Macro F1: {np.std(f1_scores):.4f}")

0:	learn: 1.7484245	total: 17.6ms	remaining: 17.5s
100:	learn: 0.5712332	total: 1.83s	remaining: 16.3s
200:	learn: 0.2736002	total: 3.83s	remaining: 15.2s
300:	learn: 0.1654347	total: 5.76s	remaining: 13.4s
400:	learn: 0.1156482	total: 7.63s	remaining: 11.4s
500:	learn: 0.0856623	total: 9.91s	remaining: 9.88s
600:	learn: 0.0659619	total: 11.9s	remaining: 7.88s
700:	learn: 0.0532351	total: 14.2s	remaining: 6.08s
800:	learn: 0.0441696	total: 16.1s	remaining: 4s
900:	learn: 0.0375979	total: 18.7s	remaining: 2.05s
999:	learn: 0.0328866	total: 21.2s	remaining: 0us
Fold Accuracy: 0.5417 | Fold F1: 0.4521
0:	learn: 1.7550686	total: 22.3ms	remaining: 22.3s
100:	learn: 0.5639833	total: 2.29s	remaining: 20.3s
200:	learn: 0.2735636	total: 5.72s	remaining: 22.8s
300:	learn: 0.1682539	total: 8.71s	remaining: 20.2s
400:	learn: 0.1162502	total: 11.1s	remaining: 16.6s
500:	learn: 0.0865162	total: 14s	remaining: 14s
600:	learn: 0.0668293	total: 17.1s	remaining: 11.3s
700:	learn: 0.0539179	total: 20.1s	